In [44]:
import pandas as pd 

spend = pd.read_csv("health_spending.csv")
life = pd.read_csv("life_expectancy.csv")
population = pd.read_csv("population.csv")

for name, df in [("spending", spend), ("life expectancy", life), ("population", population)]:
    print("=====", name, df.shape)
    print(df.head(3))
    print()

===== spending (30898, 7)
  country_name iso3_code  year indicator_code  \
0  Afghanistan       AFG  2002    che_usd2023   
1  Afghanistan       AFG  2002        ext_che   
2  Afghanistan       AFG  2002    ext_usd2023   

                    expenditure_type         value  \
0   Current Health Expenditure (CHE)  6.625590e+08   
1  External Health Expenditure (EXT)  1.373297e+01   
2  External Health Expenditure (EXT)  9.098904e+07   

                              unit  
0                constant 2023 US$  
1  % of current health expenditure  
2                constant 2023 US$  

===== life expectancy (20755, 4)
        Entity Code  Year  LifeExpectancy
0  Afghanistan  AFG  1950         27.7275
1  Afghanistan  AFG  1951         27.9634
2  Afghanistan  AFG  1952         28.4456

===== population (17394, 4)
  Country Name Country Code  Year  Value
0        Aruba          ABW  1960  54922
1        Aruba          ABW  1961  55578
2        Aruba          ABW  1962  56320



This is to identify what column can be used to link the data (is03_code, Code, Country Code all show three letter country identification codes). 

In [45]:
print(spend[["indicator_code", "expenditure_type", "unit"]].drop_duplicates())

  indicator_code                                   expenditure_type  \
0    che_usd2023                   Current Health Expenditure (CHE)   
1        ext_che                  External Health Expenditure (EXT)   
2    ext_usd2023                  External Health Expenditure (EXT)   
3      gghed_che  Domestic General Government Health Expenditure...   
4  gghed_usd2023  Domestic General Government Health Expenditure...   
5       pvtd_che        Domestic Private Health Expenditure (PVT-D)   
6   pvtd_usd2023        Domestic Private Health Expenditure (PVT-D)   

                              unit  
0                constant 2023 US$  
1  % of current health expenditure  
2                constant 2023 US$  
3  % of current health expenditure  
4                constant 2023 US$  
5  % of current health expenditure  
6                constant 2023 US$  


The code for type of expenditure matches the WHO's naming pattern. 

- che_usd2023: total health spending, in US dollars at 2023 prices.
- gghed_che: the share paid by government, as a % of the total.
- pvtd_che: the share paid privately (people and insurance), as a %.
- ext_che: the share from foreign aid, as a %.
- The _usd2023 versions of those are the same amounts in dollars rather than percentages.

I want one row per country, per year, with each measure needed as its own column. 
The spending file is in long format as each country and year appears several times with different types of expenditure. Life expectancy and population are simpler with one row per country per year. 
So I need to filter to the measures needed and then pivot from a long format to wide.

In [46]:
keep = ["che_usd2023", "gghed_che", "pvtd_che", "ext_che"]
spend_cleaned = spend[spend["indicator_code"].isin(keep)]

spend_wide = spend_cleaned.pivot_table(
    index=["iso3_code", "country_name", "year"],
    columns = "indicator_code", 
    values="value"
).reset_index()

spend_wide = spend_wide.rename(columns={
    "iso3_code" : "country_code",
    "che_usd2023": "health_spending_total_usd", 
    "gghed_che" : "gov_share_percent", 
    "pvtd_che" : "private_share_percent", 
    "ext_che" : "foreign_aid_share_percent"
})

print(spend_wide.shape)
print(spend_wide.head())
print(spend_wide[spend_wide["country_code"] == "IRL"].tail(3))

(4588, 7)
indicator_code country_code country_name  year  health_spending_total_usd  \
0                       AFG  Afghanistan  2002               6.625590e+08   
1                       AFG  Afghanistan  2003               6.818555e+08   
2                       AFG  Afghanistan  2004               7.530060e+08   
3                       AFG  Afghanistan  2005               8.540786e+08   
4                       AFG  Afghanistan  2006               9.608755e+08   

indicator_code  foreign_aid_share_percent  gov_share_percent  \
0                               13.732970           0.891424   
1                                6.650364           7.280445   
2                                9.937138           5.535272   
3                               15.710566           5.319348   
4                               18.488485           4.686539   

indicator_code  private_share_percent  
0                           85.375603  
1                           86.069191  
2                     

- Keep only the rows whose indicator is in the list I made
   - Chose che_usd2023 = to work out spending per person, by dividing the population later.
   - Chose percentages for the other three to make it comparable between countries. Dollar amounts would mostly reflect how big and rich a country is, so the percentage answers our question which is how much the person and the government pays, whatever the countrys size.  

- Pivoted the table to turn each indicator into its own column. The index columns become the row identifiers.

- Reset index is to trun these identifiers back into normal columns so the table is flat and easy to join

- Rename gives the columns names that are easy to understand. 

In [47]:
life_cleaned = life.rename(columns = {
    "Code" : "country_code",
    "Year": "year", 
    "LifeExpectancy" : "life_expectancy"
})[["country_code", "year", "life_expectancy"]]

population_cleaned = population.rename(columns = {
    "Country Code" : "country_code", 
    "Year": "year", 
    "Value" : "population"
})[["country_code", "year", "population"]]

health = (spend_wide
          .merge(life_cleaned, on=["country_code", "year"], how="inner")
          .merge(population_cleaned, on=["country_code", "year"], how="inner"))

health["spend_per_person_usd"] = health["health_spending_total_usd"] / health["population"]

print("Shape:", health.shape)
print("Min year:", health["year"].min(), "\nMax year:", health["year"].max(), "\n", health["country_code"].nunique(), "countries")
print(health[health["country_code"].isin(["IRL", "GBR", "USA"]) & (health["year"] == 2019)])

Shape: (4156, 10)
Min year: 2000 
Max year: 2021 
 193 countries
     country_code                                       country_name  year  \
1359          GBR  United Kingdom of Great Britain and Northern I...  2019   
1733          IRL                                            Ireland  2019   
3961          USA                           United States of America  2019   

      health_spending_total_usd  foreign_aid_share_percent  gov_share_percent  \
1359               3.280479e+11                   0.011181          79.731651   
1733               2.854889e+10                   0.000000          73.825516   
3961               4.190771e+12                   0.000000          51.961060   

      private_share_percent  life_expectancy  population  spend_per_person_usd  
1359              20.257166          81.7250    66628000           4923.574153  
1733              26.174486          82.2586     4976456           5736.791221  
3961              48.038937          79.1380   3302262

- Rename gives every file the same column names because merge needs the names to match.
- I joinged tables using the standard three-letter country code because country names are spelled differently across sources. 
- I used inner as it only keeps a row if that country and year exists in all the files.
  - This cuts the data to 2000-2021, because the life expectancy file stops at 2021.
  - It also drops groupings "World" or "European Union", which are in the life expectancy and population files but arent countries in the WHO data.
- I added the column "spend_per_person_usd", to divide the total by population 

In [48]:
healthy_life = pd.read_csv("hlth_hlye_linear.csv")

print(healthy_life.shape)
print(healthy_life.columns.tolist())
print(healthy_life["sex"].unique())
print(healthy_life["hlth_hle"].unique())

(17649, 11)
['DATAFLOW', 'LAST UPDATE', 'freq', 'unit', 'sex', 'hlth_hle', 'geo', 'TIME_PERIOD', 'OBS_VALUE', 'OBS_FLAG', 'CONF_STATUS']
['Females' 'Males' 'Total']
['Healthy life years in percentage of the life expectancy at birth'
 'Healthy life years in percentage of the life expectancy at 50'
 'Healthy life years in percentage of the life expectancy at 65'
 'Healthy life years in absolute value at birth'
 'Healthy life years in absolute value at 50'
 'Healthy life years in absolute value at 65'
 'Life expectancy in absolute value at birth'
 'Life expectancy in absolute value at 50'
 'Life expectancy in absolute value at 65']


In [49]:
measures = {
    "Healthy life years in absolute value at birth": "healthy_life_years",
    "Life expectancy in absolute value at birth": "life_expectancy"
}

healthy_life_small = healthy_life[healthy_life["hlth_hle"].isin(measures.keys())]

healthy_life_wide = healthy_life_small.pivot_table(
    index=["geo", "TIME_PERIOD", "sex"],
    columns="hlth_hle", 
    values="OBS_VALUE"
).reset_index()

healthy_life_wide.columns.name=None 

healthy_life_wide = healthy_life_wide.rename(columns={**measures, 
                                                      "geo":"country_name", 
                                                      "TIME_PERIOD": "year"
                                                     })
flags = (healthy_life_small
         .assign(series_break=healthy_life_small["OBS_FLAG"].fillna("").str.contains("b"))
         .groupby(["geo", "TIME_PERIOD", "sex"])["series_break"].max()
         .reset_index()
         .rename(columns={"geo":"country_name", "TIME_PERIOD": "year"}))

healthy_life_wide = healthy_life_wide.merge(flags, on=["country_name", "year", "sex"], how="left")

healthy_life_wide["unhealthy_years"] = healthy_life_wide["life_expectancy"] - healthy_life_wide["healthy_life_years"]




print(healthy_life_wide.shape)
print(healthy_life_wide[(healthy_life_wide["country_name"] == "Ireland") & (healthy_life_wide["sex"] == "Total")].tail(6))

(2028, 7)
     country_name  year    sex  healthy_life_years  life_expectancy  \
1034      Ireland  2019  Total                69.6             82.8   
1037      Ireland  2020  Total                66.2             82.6   
1040      Ireland  2021  Total                67.2             82.4   
1043      Ireland  2022  Total                66.0             82.6   
1046      Ireland  2023  Total                66.1             82.9   
1049      Ireland  2024  Total                66.9             83.1   

      series_break  unhealthy_years  
1034         False             13.2  
1037          True             16.4  
1040         False             15.2  
1043         False             16.6  
1046         False             16.8  
1049         False             16.2  


In [51]:
print(health.shape)
print(health["year"].min(), health["year"].max(), health["country_code"].nunique(), "countries")
print(health[health["country_code"].isin(["IRL", "GBR", "USA"]) & (health["year"] == 2019)]
      [["country_name", "spend_per_person_usd", "life_expectancy", "gov_share_percent"]])

(4156, 10)
2000 2021 193 countries
                                           country_name  spend_per_person_usd  \
1359  United Kingdom of Great Britain and Northern I...           4923.574153   
1733                                            Ireland           5736.791221   
3961                           United States of America          12690.605394   

      life_expectancy  gov_share_percent  
1359          81.7250          79.731651  
1733          82.2586          73.825516  
3961          79.1380          51.961060  


In [52]:
short_names = {
    "United Kingdom of Great Britain and Northern Ireland": "UK",
    "United States of America": "US",
    "Republic of Korea": "South Korea",
    "Netherlands (Kingdom of the)": "Netherlands",
    "Iran (Islamic Republic of)": "Iran",
    "Russian Federation": "Russia",
    "Venezuela (Bolivarian Republic of)": "Venezuela",
    "Bolivia (Plurinational State of)": "Bolivia",
    "Democratic Republic of the Congo": "DR Congo",
    "Lao People's Democratic Republic": "Laos",
    "Micronesia (Federated States of)": "Micronesia",
    "United Republic of Tanzania": "Tanzania",
    "Syrian Arab Republic": "Syria",
    "Republic of Moldova": "Moldova",
    "occupied Palestinian territory, including east Jerusalem": "Palestine"
}

health["label"] = health["country_name"].replace(short_names)

In [53]:
import numpy as np 

in_2019 = (health["year"] == 2019) & (health["spend_per_person_usd"] > 0) & health["life_expectancy"].notna()
x = np.log2(health.loc[in_2019, "spend_per_person_usd"])
y = health.loc[in_2019, "life_expectancy"]

slope, intercept = np.polyfit(x, y, 1)

print(f"Years gained per doubling of spending: {slope:.2f} ({in_2019.sum()} countries)")

health["expected_life_expectancy"] = np.nan
health.loc[in_2019, "expected_life_expectancy"] = (intercept + slope * x).round(2)
health["years_vs_expected"] = (health["life_expectancy"] - health["expected_life_expectancy"]).round(2)

Years gained per doubling of spending: 2.66 (192 countries)


In [ ]:
health["spend_per_person_usd"] = health["spend_per_person_usd"].round(0)
health["life_expectancy"] = health["life_expectancy"].round(1)
for col in ["gov_share_percent", "private_share_percent", "foreign_aid_share_percent"]:
    health[col] = health[col].round(1)

health.to_csv("health_main.csv", index=False)
healthy_life_wide.to_csv("healthy_life_years.csv", index=False)

print("Saved: ", health.shape, healthy_life_wide.shape, ireland_illness.shape)
print(health[health["country_code"].isin(["IRL", "GBR", "USA"]) & (health["year"] == 2019)]
      [["label", "spend_per_person_usd", "life_expectancy", "gov_share_percent"]])

Saved:  (4156, 13) (2028, 7) (13, 5)
        label  spend_per_person_usd  life_expectancy  gov_share_percent
1359       UK                4924.0             81.7               79.7
1733  Ireland                5737.0             82.3               73.8
3961       US               12691.0             79.1               52.0
